# Phase 15 — BERT-base + Focal Loss (train+valid, A100)

Fine-tuning του `bert-base-uncased` με **Focal Loss** σε train+valid.

**Γιατί Focal Loss:**
- Το dataset έχει class imbalance (π.χ. 'biological' >> 'migration')
- Focal Loss τιμωρεί περισσότερο τα εύκολα δείγματα και εστιάζει στα δύσκολα
- Συχνά βελτιώνει το macro-F1 σε imbalanced datasets

**Γιατί BERT-base:**
- 110M parameters vs 66M του DistilBERT → καλύτερες representations
- Με A100 τρέχει γρήγορα παρά το μεγαλύτερο μέγεθος

**Hyperparameters (βάσει CV):**
- LR: 2e-5 (βέλτιστο από CV για σταθερότητα)
- MAX_LENGTH: 128
- EPOCHS: 20
- Focal Loss gamma: 2.0 (standard τιμή)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

train_full = train.copy()

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

print(f'Train: {len(texts_full)}')
print(f'Test:        {len(texts_test)}')

In [ ]:
le_hazard  = LabelEncoder()
le_product = LabelEncoder()

le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes:  {n_hazard}')
print(f'Product classes: {n_product}')

# Class weights για Focal Loss — υπολογισμός από train_full
def compute_class_weights(labels, n_classes):
    counts = np.bincount(labels, minlength=n_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float)

hazard_weights  = compute_class_weights(y_full_hazard,  n_hazard).to(device)
product_weights = compute_class_weights(y_full_product, n_product).to(device)

print(f'\nHazard class weights:  {hazard_weights.cpu().numpy().round(3)}')

In [ ]:
# Focal Loss
class FocalLoss(nn.Module):
    """
    Focal Loss για class imbalance.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    gamma=0 → κανονικό Cross Entropy
    gamma=2 → standard Focal Loss (Lin et al. 2017)
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt      = torch.exp(-ce_loss)          # p_t = probability of correct class
        focal   = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()


hazard_criterion  = FocalLoss(gamma=2.0, weight=hazard_weights)
product_criterion = FocalLoss(gamma=2.0, weight=product_weights)
print('Focal Loss initialized! (gamma=2.0)')

In [ ]:
MODEL_NAME   = 'bert-base-uncased'
BATCH_SIZE   = 32   
MAX_LENGTH   = 128
LR           = 2e-5
N_EPOCHS     = 20
WARMUP_RATIO = 0.1

print(f'Φόρτωση tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

dummy_labels = np.zeros(len(texts_test), dtype=int)

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = model(**batch).logits
        loss   = criterion(logits, labels)  # Focal Loss αντί Cross Entropy
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

In [ ]:
full_loader_hazard  = DataLoader(FoodHazardDataset(texts_full, y_full_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
full_loader_product = DataLoader(FoodHazardDataset(texts_full, y_full_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
test_loader_hazard  = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(full_loader_hazard)}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {N_EPOCHS} | LR: {LR}')

## Εκπαίδευση BERT-base — Hazard Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ BERT-base + Focal Loss — HAZARD ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS} | LR: {LR}\n')

bert_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_hazard
).to(device)

optimizer_hazard = AdamW(bert_hazard.parameters(), lr=LR, weight_decay=0.01)
total_steps      = len(full_loader_hazard) * N_EPOCHS
warmup_steps     = int(total_steps * WARMUP_RATIO)
scheduler_hazard = get_linear_schedule_with_warmup(
    optimizer_hazard,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_hazard, full_loader_hazard, optimizer_hazard, scheduler_hazard, hazard_criterion)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Focal Loss: {loss:.4f}')

print('\nΕκπαίδευση Hazard ολοκληρώθηκε')

## Εκπαίδευση BERT-base — Product Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ BERT-base + Focal Loss — PRODUCT ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS} | LR: {LR}\n')

bert_product = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_product
).to(device)

optimizer_product = AdamW(bert_product.parameters(), lr=LR, weight_decay=0.01)
total_steps       = len(full_loader_product) * N_EPOCHS
warmup_steps      = int(total_steps * WARMUP_RATIO)
scheduler_product = get_linear_schedule_with_warmup(
    optimizer_product,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_product, full_loader_product, optimizer_product, scheduler_product, product_criterion)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Focal Loss: {loss:.4f}')

print('\nΕκπαίδευση Product ολοκληρώθηκε')

In [ ]:
# Δημιουργία validation DataLoaders
def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

valid_loader_hazard = DataLoader(
    FoodHazardDataset(
        make_text(valid),
        le_hazard.transform(valid['hazard-category']),
        tokenizer, MAX_LENGTH
    ),
    batch_size=BATCH_SIZE, shuffle=False
)

valid_loader_product = DataLoader(
    FoodHazardDataset(
        make_text(valid),
        le_product.transform(valid['product-category']),
        tokenizer, MAX_LENGTH
    ),
    batch_size=BATCH_SIZE, shuffle=False
)

# Αποθήκευση valid probabilities
np.save('bertbase_focal_trainonly_valid_hazard_probs.npy',
        get_probabilities(bert_hazard,  valid_loader_hazard))
np.save('bertbase_focal_trainonly_valid_product_probs.npy',
        get_probabilities(bert_product, valid_loader_product))

print('Valid probs αποθηκεύτηκαν')

## Αποθήκευση Probabilities + Submission

In [ ]:
# Test probabilities για ensemble
bert_test_hazard_probs  = get_probabilities(bert_hazard,  test_loader_hazard)
bert_test_product_probs = get_probabilities(bert_product, test_loader_product)

np.save('bertbase_focal_test_hazard_probs.npy',  bert_test_hazard_probs)
np.save('bertbase_focal_test_product_probs.npy', bert_test_product_probs)

print(f'Test hazard probs:  {bert_test_hazard_probs.shape}')
print(f'Test product probs: {bert_test_product_probs.shape}')
print('Αποθηκεύτηκαν τα .npy για ensemble')

# Submission BERT-base μόνο
test_hazard  = le_hazard.inverse_transform(bert_test_hazard_probs.argmax(axis=1))
test_product = le_product.inverse_transform(bert_test_product_probs.argmax(axis=1))

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_bertbase_focal.csv', index=False)

print('Αποθηκεύτηκε: submission_bertbase_focal.csv')
print('\n=== ΣΥΓΚΡΙΣΗ ===')
print('Best so far:   0.76064  (DistilBERT train+valid 20 epochs)')

In [ ]:
# Αποθήκευση μοντέλων για μελλοντική χρήση
torch.save(bert_hazard.state_dict(),  'bertbase_focal_hazard_weights.pt')
torch.save(bert_product.state_dict(), 'bertbase_focal_product_weights.pt')
print('Model weights αποθηκεύτηκαν!')